# Structured Perceptron for NER

We train a structured perceptron on the CoNLL 2002 Spanish dataset to perform Named Entity Recognition (NER).

The structured perceptron extends the standard perceptron to **sequence labeling**: instead of predicting a single label, we predict the best sequence of labels jointly.

The three key ingredients are:
1. A **feature function** `φ(x, i, y_prev, y)` that returns the complete joint features for position `i`
2. A **Viterbi decoder** that finds the highest-scoring label sequence given the current weights
3. A **perceptron update** that nudges the weights when the prediction is wrong

## Load data

In [ ]:
import nltk
nltk.download('conll2002', quiet=True)

train_sents = list(nltk.corpus.conll2002.iob_sents('esp.train'))
test_sents  = list(nltk.corpus.conll2002.iob_sents('esp.testa'))

# Each sentence is a list of (word, pos, iob) tuples
print(train_sents[0])

In [ ]:
def sent_to_XY(sent):
    """Split a CoNLL sentence into a list of tokens and a list of labels."""
    tokens = [(word, pos) for word, pos, _ in sent]
    labels = [iob for _, _, iob in sent]
    return tokens, labels

X_train, y_train = zip(*[sent_to_XY(s) for s in train_sents])
X_test,  y_test  = zip(*[sent_to_XY(s) for s in test_sents])

# All possible labels
label_set = sorted(set(label for seq in y_train for label in seq))
print(label_set)

## Feature functions

A feature function takes a token sequence, a position, the previous label and the current label, and returns a list of **complete joint feature strings**.

Each feature string fully identifies a weight in the model.

We provide two versions:
- `token_features_hmm`: word identity and transition only (equivalent to an HMM)
- `token_features`: richer surface features that go beyond what an HMM can express

In [ ]:
import pandas as pd

def token_features_hmm(tokens, i, prev_label, label):
    """Minimal features: word identity and transition only. Equivalent to an HMM."""
    word, pos = tokens[i]
    feats = [
        f'word={word.lower()}::{label}',
        f'transition:{prev_label}->{label}',
    ]
    if i == len(tokens) - 1:
        feats.append(f'transition:{label}-><STOP>')

    return feats

def token_features(tokens, i, prev_label, label):
    """Richer surface features: suffixes, capitalization, context window."""
    word, pos = tokens[i]
    feats = [
        f'word={word.lower()}::{label}',
        f'suffix2={word[-2:].lower()}::{label}',
        f'suffix3={word[-3:].lower()}::{label}',
        f'pos={pos}::{label}',
        f'is_upper={word[0].isupper()}::{label}',
        f'is_all_upper={word.isupper()}::{label}',
        f'transition:{prev_label}->{label}',
    ]
    if i > 0:
        prev_word, prev_pos = tokens[i - 1]
        feats += [
            f'prev_word={prev_word.lower()}::{label}',
            f'prev_pos={prev_pos}::{label}',
        ]

    if i < len(tokens) - 1:
        next_word, next_pos = tokens[i + 1]
        feats += [
            f'next_word={next_word.lower()}::{label}',
            f'next_pos={next_pos}::{label}',
        ]
    else:
        feats.append(f'transition:{label}-><STOP>')

    return feats


def show_features(tokens, labels, feat_fn):
    rows = []
    prev_label = '<START>'
    for i, ((word, pos), label) in enumerate(zip(tokens, labels)):
        for feat in feat_fn(tokens, i, prev_label, label):
            rows.append({'i': i, 'word': word, 'prev_label': prev_label, 'label': label, 'feature': feat})
        prev_label = label
    return pd.DataFrame(rows)

## Scoring

The score of a set of joint features is simply the sum of their weights:

$$s(\phi) = \sum_{f \in \phi} w_f$$

In [ ]:
def score(weights, feats):
    return sum(weights[f] for f in feats)

## Viterbi decoding

Given the current weights, find the highest-scoring label sequence for a sentence:

$$\hat{y} = \arg\max_y \sum_{i} s\left(\phi(x, i, y_{i-1}, y_i)\right)$$

This is exactly the same Viterbi algorithm you saw for HMMs. The only difference is that we use **linear scores** instead of log-probabilities.

In [ ]:
def viterbi(weights, tokens, label_set, feat_fn):
    n = len(tokens)
    # viterbi_scores[i][y] = best score for sequences ending with label y at position i
    viterbi_scores = [{}]
    backpointer    = [{}]

    # Initialize: position 0, no previous label
    for label in label_set:
        viterbi_scores[0][label] = score(weights, feat_fn(tokens, 0, '<START>', label))
        backpointer[0][label]    = '<START>'

    # Fill
    for i in range(1, n):
        viterbi_scores.append({})
        backpointer.append({})
        for label in label_set:
            best_prev, best_score = max(
                ((prev,
                  viterbi_scores[i-1][prev] + score(weights, feat_fn(tokens, i, prev, label)))
                 for prev in label_set),
                key=lambda x: x[1]
            )
            viterbi_scores[i][label] = best_score
            backpointer[i][label]    = best_prev

    # Backtrack
    best_last = max(label_set, key=lambda y: viterbi_scores[n-1][y])
    sequence  = [best_last]
    for i in range(n - 1, 0, -1):
        sequence.append(backpointer[i][sequence[-1]])
    return list(reversed(sequence))

## Structured Perceptron

The update rule is:

$$w \leftarrow w + \phi(x, y^*) - \phi(x, \hat{y})$$

where $y^*$ is the gold sequence and $\hat{y}$ is the Viterbi prediction.  
We increment weights for gold features and decrement weights for predicted features.

In [ ]:
import random
from collections import defaultdict

def update(weights, tokens, y_true, y_pred, feat_fn):
    """Perceptron update: reward gold features, penalise predicted features."""
    prev_true = prev_pred = '<START>'
    for i, (gold, pred) in enumerate(zip(y_true, y_pred)):
        if gold != pred or prev_true != prev_pred:
            for f in feat_fn(tokens, i, prev_true, gold):
                weights[f] += 1
            for f in feat_fn(tokens, i, prev_pred, pred):
                weights[f] -= 1
        prev_true = gold
        prev_pred = pred


def train(X_train, y_train, label_set, feat_fn, n_epochs=5, seed=42):
    weights = defaultdict(float)
    indices = list(range(len(X_train)))
    random.seed(seed)

    for epoch in range(n_epochs):
        random.shuffle(indices)
        errors = 0
        for i in indices:
            y_pred = viterbi(weights, X_train[i], label_set, feat_fn)
            if y_pred != list(y_train[i]):
                update(weights, X_train[i], y_train[i], y_pred, feat_fn)
                errors += 1
        print(f'Epoch {epoch+1} - sentences with errors: {errors}/{len(X_train)}')

    return weights

## Train and evaluate

Switch between `token_features_hmm` and `token_features` to compare both models.

In [ ]:
#feat_fn = token_features_hmm
feat_fn = token_features

In [ ]:
# Example: take the first sentence from the test set
tokens, labels = X_test[0], y_test[0]
print(show_features(tokens, labels, feat_fn=feat_fn).to_string(index=False))

In [ ]:
weights = train(X_train, y_train, label_set, feat_fn=feat_fn, n_epochs=5)

In [ ]:
from sklearn.metrics import classification_report

y_pred_test = [viterbi(weights, tokens, label_set, feat_fn=feat_fn) for tokens in X_test]

y_true_flat = [label for seq in y_test      for label in seq]
y_pred_flat = [label for seq in y_pred_test for label in seq]

print(classification_report(y_true_flat, y_pred_flat, labels=label_set, digits=3, zero_division=0))

In [ ]:
# Exclude 'O' labels
labels = [l for l in label_set if l != 'O']
print(classification_report(y_true_flat, y_pred_flat, labels=labels, digits=3, zero_division=0))

## Inspect the model

One advantage of linear models: the weights are directly interpretable.  
Each key in the weight dictionary is a complete joint feature string.

In [ ]:
def top_features(weights, label, n=10):
    relevant = {f: w for f, w in weights.items() if f.endswith(f'::{label}') and w > 0}
    top = sorted(relevant.items(), key=lambda x: x[1], reverse=True)[:n]
    return pd.DataFrame(top, columns=['feature', 'weight'])

print('=== Top features for B-PER ===')
print(top_features(weights, 'B-PER'))

print('\n=== Top features for B-LOC ===')
print(top_features(weights, 'B-LOC'))